# MediAssist Capstone Project

In [ ]:
# Installing required libraries

!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-groq
!pip install -q langgraph
!pip install -q pandas
!pip install -q python-dotenv
!pip install -q langchain-huggingface
!pip install -q langchain-text-splitters

In [ ]:
# Defining Agents

import os

from memory_store import retrieve_similar_cases, save_case
from langchain_groq import ChatGroq 
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file


llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="llama-3.3-70b-versatile",  # Updated model name
    temperature=0.2,
    max_tokens=1024
)


def intake_agent(state):
    return state


def history_agent(state):
    prompt = f"Analyze medical history risks: {state['history']}"
    response = llm.invoke(prompt)
    return {"history_analysis": response.content}


def symptom_agent(state):
    prompt = f"Analyze emergency symptoms: {state['symptoms']}"
    response = llm.invoke(prompt)
    return {"symptoms_analysis": response.content}


def lab_agent(state):
    prompt = f"Analyze patient lab values: {state['lab_results']}"
    response = llm.invoke(prompt)
    return {"lab_analysis": response.content}


def memory_agent(state):
    memory = retrieve_similar_cases(state['symptoms'])
    return {"memory_context": memory}


def risk_agent(state):
    score = 0
    flags = []

    symptoms = state['symptoms'].lower()
    labs = state['lab_results']

    if 'chest pain' in symptoms:
        score += 40
        flags.append('Possible cardiac issue')

    if labs.get('oxygen', 100) < 90:
        score += 30
        flags.append('Low oxygen')

    if labs.get('troponin', 0) > 0.4:
        score += 40
        flags.append('Elevated troponin')

    return {
        "risk_score": score,
        "risk_flags": flags
    }


def triage_agent(state):
    score = state['risk_score']

    if score >= 80:
        priority = 'P1'
        confidence = 0.97
    elif score >= 60:
        priority = 'P2'
        confidence = 0.91
    elif score >= 30:
        priority = 'P3'
        confidence = 0.82
    else:
        priority = 'P4'
        confidence = 0.70

    return {
        "priority": priority,
        "confidence": confidence
    }


def review_agent(state):
    reasoning = f"""
Priority: {state['priority']}
Confidence: {state['confidence']}
Flags: {state['risk_flags']}

History Analysis:
{state.get('history_analysis')}

Symptoms Analysis:
{state.get('symptoms_analysis')}

Lab Analysis:
{state.get('lab_analysis')}

Memory Context:
{state.get('memory_context')}
"""

    save_case(state['patient_id'], reasoning)

    return {
        "reasoning": reasoning
    }


c:\Users\v-vyvyas\AppData\Local\anaconda3\envs\aiagent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4541.39it/s]
c:\Users\v-vyvyas\capstone-agentic-ai\memory_store.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [ ]:
# Defining Memory Store

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )

vectorstore = Chroma(
    collection_name="hospital_cases",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)


def retrieve_similar_cases(query: str):
    docs = vectorstore.similarity_search(query, k=3)
    return "\n".join([d.page_content for d in docs])


def save_case(case_id: str, summary: str):
    vectorstore.add_texts(
        texts=[summary],
        metadatas=[{"case_id": case_id}]
    )


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6232.07it/s]


In [ ]:
# Defining Workflow

from langgraph.graph import StateGraph, END
from state import TriageState

from agents import (
    intake_agent,
    history_agent,
    symptom_agent,
    lab_agent,
    memory_agent,
    risk_agent,
    triage_agent,
    review_agent
)

workflow = StateGraph(TriageState)

workflow.add_node('intake', intake_agent)
workflow.add_node('history', history_agent)
workflow.add_node('symptom', symptom_agent)
workflow.add_node('lab', lab_agent)
workflow.add_node('memory', memory_agent)
workflow.add_node('risk', risk_agent)
workflow.add_node('triage', triage_agent)
workflow.add_node('review', review_agent)

workflow.set_entry_point('intake')

workflow.add_edge('intake', 'history')
workflow.add_edge('intake', 'symptom')
workflow.add_edge('intake', 'lab')
workflow.add_edge('intake', 'memory')

workflow.add_edge('history', 'risk')
workflow.add_edge('symptom', 'risk')
workflow.add_edge('lab', 'risk')
workflow.add_edge('memory', 'risk')

workflow.add_edge('risk', 'triage')
workflow.add_edge('triage', 'review')
workflow.add_edge('review', END)

app = workflow.compile()


In [ ]:
# Running the Workflow

from graph import app

patient_case = {
    "patient_id": "P1001",
    "age": 72,
    "history": ["diabetes", "hypertension"],
    "symptoms": "Chest pain and shortness of breath",
    "lab_results": {
        "troponin": 0.8,
        "oxygen": 87,
        "bp": "180/120"
    }
}

result = app.invoke(patient_case)

print("""
=== TRIAGE RESULT ===
""")
print('Priority:', result['priority'])
print('Confidence:', result['confidence'])
print("""
Reasoning:
""")
print(result['reasoning'])



=== TRIAGE RESULT ===

Priority: P1
Confidence: 0.97

Reasoning:


Priority: P1
Confidence: 0.97
Flags: ['Possible cardiac issue', 'Low oxygen', 'Elevated troponin']

History Analysis:
**Medical History Risks Analysis**

The provided medical history risks are:

1. **Diabetes**
2. **Hypertension**

**Risk Assessment:**

Both diabetes and hypertension are significant health concerns that can increase the risk of developing various complications, particularly cardiovascular diseases.

* **Diabetes:**
	+ Increases the risk of heart disease, stroke, and kidney disease
	+ Can cause nerve damage (neuropathy), vision problems, and foot ulcers
	+ Requires ongoing management and monitoring to control blood sugar levels
* **Hypertension:**
	+ Increases the risk of heart disease, stroke, and kidney disease
	+ Can cause cardiovascular damage, heart failure, and vision problems
	+ Requires ongoing management and monitoring to control blood pressure levels

**Combined Risk:**

The presence of both d